# Sistema de Detecção de Obstáculos para Apoio a Deficientes Visuais

Utilizando YOLOv8 para Análise em Tempo Real e Geração de Feedback Auditivo

### Configuração do Ambiente e Instalação de Dependências
Esta célula prepara o ambiente do Google Colab, instalando todas as bibliotecas e pacotes de sistema necessários para a execução do modelo e da interface de áudio.

In [ ]:
# Instalar as bibliotecas necessárias
!pip install opencv-python
!pip install ultralytics
!pip install roboflow
!pip install pyttsx3
# Instalação das Dependências para Áudio no Ambiente Colab(Linux)
# gTTS para salvar os alertas em MP3 e FFmpeg para unir áudio e vídeo
!pip install gtts
!apt-get update && apt-get install -y espeak-ng
!apt-get install -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 72.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 http://archive.ubuntu.com/ubun

### Download e Preparação do Dataset via Roboflow
Aqui, o código se conecta à plataforma Roboflow para baixar e descompactar o dataset de imagens. O processo já organiza os arquivos na estrutura de pastas exigida pelo YOLOv8, incluindo o arquivo de configuração data.yaml.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="rzm6rHjTO1nFDyWOWKBG")
project = rf.workspace("amanda-ow7bi").project("multiplos-objetos-2-23osn")
version = project.version(3)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to multiplos-objetos-2-3 in yolov8:: 100%|██████████| 20738/20738 [00:06<00:00, 3447.13it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Treinamento do Modelo YOLOv8
Esta célula executa o processo de treinamento do modelo. Utilizando a técnica de Transfer Learning, o modelo yolov8n pré-treinado é ajustado com nosso dataset por 60 épocas. Ao final, os pesos do modelo com o melhor desempenho são salvos no arquivo best.pt.

In [ ]:
# Importar a classe YOLO
from ultralytics import YOLO

# Carregar o modelo base do YOLOv8
# Usaremos o modelo "nano" que é o mais rápido
model = YOLO('yolov8n.pt')

# Treinar o modelo com o seu dataset
# O caminho para o dataset é fornecido pela variável 'dataset.location'
# gerada no passo anterior.
results = model.train(data=f"{dataset.location}/data.yaml", epochs=60)

Ultralytics 8.3.201 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/multiplos-objetos-2-3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=

KeyboardInterrupt: 

### Código Teste do Modelo com Imagens e Vídeos:
Este código permite testar o modelo treinado de forma interativa. Fazendo o upload de um arquivo de imagem ou vídeo para ver a detecção de obstáculos em ação.

Funcionalidades:

* Upload de Arquivos: Permite o envio de uma imagem (.jpg, .png) ou um vídeo (.mp4) do seu computador.

* Processamento de Imagem: Se uma imagem for enviada, o sistema a exibirá na tela com os obstáculos detectados e demarcados.

* Processamento de Vídeo: Se um vídeo for enviado, o sistema irá:

* Processar cada frame para adicionar as detecções visuais.

* Gerar e salvar um arquivo de áudio para cada alerta.

* Unir o vídeo e os áudios em um único arquivo final.

* Download Automático: Ao final do processamento do vídeo, o download do arquivo resultante (video_final_com_audio.mp4) será iniciado automaticamente no seu navegador.

In [ ]:
# Passo 1: Importar tudo que vamos usar
import cv2
from ultralytics import YOLO
import numpy as np
import time
from gtts import gTTS
from google.colab import files
from google.colab.patches import cv2_imshow
import os
import subprocess

# --- Configuração do Modelo ---
# O arquivo 'best.pt' precisa estar presente no colab
model = YOLO('best.pt')

def processar_imagem(caminho_imagem):
    print(f"Processando a imagem: {caminho_imagem}")
    frame = cv2.imread(caminho_imagem)
    results = model(frame)
    for box in results[0].boxes:
        confianca = box.conf[0]
        if confianca >= 0.6:
            coords = box.xyxy[0].cpu().numpy().astype(int)
            x1, y1, x2, y2 = coords
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            id_classe = int(box.cls[0])
            nome_classe = model.names[id_classe]
            rotulo = f'{nome_classe} {confianca:.2f}'
            cv2.putText(frame, rotulo, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    print("Exibindo imagem com detecções:")
    cv2_imshow(frame)

def processar_video(caminho_video):
    """
    Processa um vídeo, RASTREIA objetos, desenha as detecções,
    gera alertas de áudio por objeto individual e salva o resultado final.
    """
    print(f"Iniciando processamento com rastreamento do vídeo: {caminho_video}")
    cap = cv2.VideoCapture(caminho_video)

    # Configurações para salvar o vídeo de saída
    largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    caminho_video_sem_audio = 'video_sem_audio.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(caminho_video_sem_audio, fourcc, fps, (largura, altura))

    # --- LÓGICA DE ÁUDIO APRIMORADA COM RASTREAMENTO ---
    # Dicionário para rastrear o último alerta de áudio POR ID DE OBJETO
    objetos_rastreados_audio = {}
    COOLDOWN_SEGUNDOS = 8

    lista_de_alertas = []
    contador_alertas = 0

    print("Etapa 1 de 2: Analisando e rastreando objetos no vídeo...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        timestamp_atual = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

        # USAMOS O RASTREADOR
        results = model.track(frame, persist=True)

        # Pega os IDs de todos os objetos rastreados no frame atual
        ids_detectados_no_frame = []
        if results[0].boxes.id is not None:
            ids_detectados_no_frame = results[0].boxes.id.int().cpu().tolist()

        # Itera sobre cada caixa (box) detectada no resultado
        for box in results[0].boxes:
            confianca = box.conf[0]

            # Continua apenas se a confiança for boa e o objeto tiver um ID de rastreamento
            if confianca >= 0.6 and box.id is not None:
                id_classe = int(box.cls[0])
                nome_classe = model.names[id_classe]
                track_id = int(box.id[0]) # Pega o ID único do objeto

                # Lógica de áudio baseada no ID do objeto
                if track_id not in objetos_rastreados_audio or timestamp_atual - objetos_rastreados_audio[track_id] > COOLDOWN_SEGUNDOS:
                    coords = box.xyxy[0].cpu().numpy().astype(int)
                    x1, y1, x2, y2 = coords
                    centro_x_objeto = (x1 + x2) / 2
                    posicao = "à sua esquerda" if centro_x_objeto < largura / 3 else "à sua direita" if centro_x_objeto > largura * (2/3) else "à sua frente"

                    mensagem = f"{nome_classe} {posicao}"

                    tts = gTTS(mensagem, lang='pt-br')
                    caminho_audio = f"alerta_{contador_alertas}.mp3"
                    tts.save(caminho_audio)
                    lista_de_alertas.append((timestamp_atual, caminho_audio))
                    contador_alertas += 1

                    # Atualiza o tempo do último alerta para ESTE ID específico
                    objetos_rastreados_audio[track_id] = timestamp_atual

                # Desenha a caixa e o rótulo (agora com o ID) no frame
                coords = box.xyxy[0].cpu().numpy().astype(int)
                x1, y1, x2, y2 = coords
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                rotulo = f'ID:{track_id} {nome_classe} {confianca:.2f}'
                cv2.putText(frame, rotulo, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Limpa da memória de áudio os IDs que não estão mais na tela
        ids_para_remover = [id_obj for id_obj in objetos_rastreados_audio if id_obj not in ids_detectados_no_frame]
        for id_obj in ids_para_remover:
            del objetos_rastreados_audio[id_obj]

        out.write(frame)

    cap.release()
    out.release()
    print("Etapa 1 concluída.")

    # --- ETAPA 2: Unir vídeo e áudios com FFmpeg ---
    print("\nEtapa 2 de 2: Unindo vídeo e áudio com FFmpeg...")
    if not lista_de_alertas:
        print("Nenhum alerta de áudio foi gerado. O vídeo final não terá som.")
        caminho_video_final = caminho_video_sem_audio
    else:
        caminho_video_final = 'video_final_com_audio.mp4'
        comando_ffmpeg = ['ffmpeg', '-y', '-i', caminho_video_sem_audio]
        for _, caminho_audio in lista_de_alertas:
            comando_ffmpeg.extend(['-i', caminho_audio])
        filter_complex = ""
        for i, (timestamp, _) in enumerate(lista_de_alertas):
            delay_ms = int(timestamp * 1000)
            filter_complex += f"[{i+1}:a]adelay={delay_ms}|adelay={delay_ms}[a{i}];"
        filter_complex += "".join([f"[a{i}]" for i in range(len(lista_de_alertas))])
        filter_complex += f"amix=inputs={len(lista_de_alertas)}[a]"
        comando_ffmpeg.extend(['-filter_complex', filter_complex, '-map', '0:v', '-map', '[a]', '-c:v', 'copy', '-c:a', 'aac', '-shortest', caminho_video_final])
        subprocess.run(comando_ffmpeg, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("Etapa 2 concluída.")

    # --- Limpeza e Download ---
    os.remove(caminho_video_sem_audio)
    for _, caminho_audio in lista_de_alertas:
        os.remove(caminho_audio)
    print("Arquivos temporários removidos.")
    files.download(caminho_video_final)

# --- ROTINA PRINCIPAL ---
print("Por favor, selecione um arquivo de imagem (jpg, png) ou vídeo (mp4) para fazer o upload.")
uploaded = files.upload()

if not uploaded:
    print("\nNenhum arquivo foi enviado. Tente novamente.")
else:
    nome_arquivo = list(uploaded.keys())[0]
    extensao = os.path.splitext(nome_arquivo)[1].lower()

    if extensao in ['.jpg', '.jpeg', '.png']:
        processar_imagem(nome_arquivo)
    elif extensao in ['.mp4', '.avi', '.mov']:
        processar_video(nome_arquivo)
    else:
        print(f"Erro: Formato de arquivo '{extensao}' não suportado. Por favor, envie uma imagem ou vídeo.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.
Por favor, selecione um arquivo de imagem (jpg, png) ou vídeo (mp4) para fazer o upload.


Saving Como Um Deficiente Visual Anda Na Rua_ Me Acompanhe Pelas Calçadas E Passarelas Do Meu Caminho(720P_HD)_1.mp4 to Como Um Deficiente Visual Anda Na Rua_ Me Acompanhe Pelas Calçadas E Passarelas Do Meu Caminho(720P_HD)_1 (5).mp4
Iniciando processamento com rastreamento do vídeo: Como Um Deficiente Visual Anda Na Rua_ Me Acompanhe Pelas Calçadas E Passarelas Do Meu Caminho(720P_HD)_1 (5).mp4
Etapa 1 de 2: Analisando e rastreando objetos no vídeo...

0: 384x640 (no detections), 7.8ms
Speed: 2.1ms preprocess, 7.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 7.6ms
Speed: 3.4ms preprocess, 7.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 8.1ms
Speed: 2.6ms preprocess, 8.1ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 11.7ms
Speed: 2.4ms preprocess, 11.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no d


0: 384x640 1 carro, 8.1ms
Speed: 3.4ms preprocess, 8.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.6ms
Speed: 2.5ms preprocess, 8.6ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.3ms
Speed: 2.3ms preprocess, 8.3ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 9.2ms
Speed: 2.4ms preprocess, 9.2ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.8ms
Speed: 4.0ms preprocess, 8.8ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.0ms
Speed: 2.3ms preprocess, 8.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.9ms
Speed: 2.8ms preprocess, 8.9ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 9.4ms
Speed: 3.1ms preprocess, 9.4ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no 


0: 384x640 1 carro, 8.4ms
Speed: 2.5ms preprocess, 8.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 9.1ms
Speed: 2.3ms preprocess, 9.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 9.1ms
Speed: 2.2ms preprocess, 9.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.0ms
Speed: 2.2ms preprocess, 8.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 6.7ms
Speed: 2.1ms preprocess, 6.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.4ms
Speed: 2.1ms preprocess, 8.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.6ms
Speed: 2.0ms preprocess, 8.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 7.3ms
Speed: 2.0ms preprocess, 7.3ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 ca


0: 384x640 (no detections), 8.2ms
Speed: 4.4ms preprocess, 8.2ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 7.4ms
Speed: 3.3ms preprocess, 7.4ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 7.3ms
Speed: 2.4ms preprocess, 7.3ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 7.1ms
Speed: 3.3ms preprocess, 7.1ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 7.7ms
Speed: 3.3ms preprocess, 7.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 13.5ms
Speed: 4.0ms preprocess, 13.5ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)



0: 384x640 1 escada, 10.0ms
Speed: 2.1ms preprocess, 10.0ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 9.2ms
Speed: 2.0ms preprocess, 9.2ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 7.5ms
Speed: 2.0ms preprocess, 7.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 8.3ms
Speed: 2.4ms preprocess, 8.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 8.1ms
Speed: 2.4ms preprocess, 8.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 7.2ms
Speed: 2.1ms preprocess, 7.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 9.8ms
Speed: 2.5ms preprocess, 9.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 escada, 7.3ms
Speed: 2.1ms preprocess, 7.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 38

... (32 outputs left out)


0: 384x640 1 carro, 6.9ms
Speed: 2.1ms preprocess, 6.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 12.8ms
Speed: 2.1ms preprocess, 12.8ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.8ms
Speed: 2.1ms preprocess, 8.8ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.0ms
Speed: 2.0ms preprocess, 8.0ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 9.2ms
Speed: 2.2ms preprocess, 9.2ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 12.0ms
Speed: 2.0ms preprocess, 12.0ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.7ms
Speed: 2.2ms preprocess, 8.7ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.8ms
Speed: 2.0ms preprocess, 8.8ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 


0: 384x640 1 carro, 12.0ms
Speed: 2.0ms preprocess, 12.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 10.4ms
Speed: 2.0ms preprocess, 10.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.8ms
Speed: 2.0ms preprocess, 8.8ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.9ms
Speed: 2.0ms preprocess, 8.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 8.6ms
Speed: 1.9ms preprocess, 8.6ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 9.7ms
Speed: 2.3ms preprocess, 9.7ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 11.1ms
Speed: 2.0ms preprocess, 11.1ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 carro, 7.9ms
Speed: 1.9ms preprocess, 7.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x64

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>